In [0]:
print("test")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime, timezone
from urllib.parse import urlencode
import requests
import json
import hashlib
import uuid
import time

BRONZE_SCHEMA = "supplysage_bronze"

INGESTION_BATCH_ID = f"ext_api_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{str(uuid.uuid4())[:8]}"
INGESTION_VERSION = "v0"
CREATED_BY = "Vigya"
NOTEBOOK_NAME = "05_bronze_ingest_external_risk_sources_v0"

USER_AGENT = "SupplySageAI/0.1 (vigya.awasthi@tamu.edu)"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")

print("INGESTION_BATCH_ID:", INGESTION_BATCH_ID)
print("BRONZE_SCHEMA:", BRONZE_SCHEMA)
print("NOTEBOOK_NAME:", NOTEBOOK_NAME)

In [0]:
try:
    EIA_API_KEY = dbutils.secrets.get(scope="supplysage", key="eia_api_key")
    print("EIA API key found.")
    print("Masked EIA key:", "*" * max(len(EIA_API_KEY) - 4, 0) + EIA_API_KEY[-4:])
except Exception as e:
    EIA_API_KEY = None
    print("EIA API key not found. EIA ingestion will be skipped.")
    print(str(e))

In [0]:
def now_utc():
    return datetime.now(timezone.utc).isoformat()


def json_dumps_safe(value):
    return json.dumps(value, default=str, ensure_ascii=False)


def sha256_text(value):
    if value is None:
        value = ""
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


EXTERNAL_RAW_SCHEMA = T.StructType([
    T.StructField("api_run_id", T.StringType(), True),
    T.StructField("ingestion_batch_id", T.StringType(), True),
    T.StructField("source_name", T.StringType(), True),
    T.StructField("endpoint_name", T.StringType(), True),
    T.StructField("request_url", T.StringType(), True),
    T.StructField("query_params_json", T.StringType(), True),
    T.StructField("request_headers_json", T.StringType(), True),
    T.StructField("response_status_code", T.IntegerType(), True),
    T.StructField("response_ok", T.BooleanType(), True),
    T.StructField("fetched_at", T.StringType(), True),
    T.StructField("raw_payload_json", T.StringType(), True),
    T.StructField("payload_hash", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("ingestion_version", T.StringType(), True),
    T.StructField("created_by", T.StringType(), True),
    T.StructField("notebook_name", T.StringType(), True)
])


API_RUN_SCHEMA = T.StructType([
    T.StructField("api_run_id", T.StringType(), True),
    T.StructField("ingestion_batch_id", T.StringType(), True),
    T.StructField("source_name", T.StringType(), True),
    T.StructField("endpoint_name", T.StringType(), True),
    T.StructField("request_url", T.StringType(), True),
    T.StructField("response_status_code", T.IntegerType(), True),
    T.StructField("response_ok", T.BooleanType(), True),
    T.StructField("fetched_at", T.StringType(), True),
    T.StructField("payload_hash", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("ingestion_version", T.StringType(), True),
    T.StructField("created_by", T.StringType(), True),
    T.StructField("notebook_name", T.StringType(), True)
])


def normalize_external_row(row):
    return {
        "api_run_id": str(row.get("api_run_id")) if row.get("api_run_id") is not None else None,
        "ingestion_batch_id": str(row.get("ingestion_batch_id")) if row.get("ingestion_batch_id") is not None else None,
        "source_name": str(row.get("source_name")) if row.get("source_name") is not None else None,
        "endpoint_name": str(row.get("endpoint_name")) if row.get("endpoint_name") is not None else None,
        "request_url": str(row.get("request_url")) if row.get("request_url") is not None else None,
        "query_params_json": str(row.get("query_params_json")) if row.get("query_params_json") is not None else None,
        "request_headers_json": str(row.get("request_headers_json")) if row.get("request_headers_json") is not None else None,
        "response_status_code": int(row.get("response_status_code")) if row.get("response_status_code") is not None else None,
        "response_ok": bool(row.get("response_ok")) if row.get("response_ok") is not None else None,
        "fetched_at": str(row.get("fetched_at")) if row.get("fetched_at") is not None else None,
        "raw_payload_json": str(row.get("raw_payload_json")) if row.get("raw_payload_json") is not None else None,
        "payload_hash": str(row.get("payload_hash")) if row.get("payload_hash") is not None else None,
        "error_message": str(row.get("error_message")) if row.get("error_message") is not None else None,
        "ingestion_version": str(row.get("ingestion_version")) if row.get("ingestion_version") is not None else None,
        "created_by": str(row.get("created_by")) if row.get("created_by") is not None else None,
        "notebook_name": str(row.get("notebook_name")) if row.get("notebook_name") is not None else None
    }


def safe_get_json(source_name, endpoint_name, url, params=None, headers=None, timeout=60):
    if params is None:
        params = {}

    if headers is None:
        headers = {}

    final_headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/json"
    }
    final_headers.update(headers)

    request_url = url
    if params:
        request_url = f"{url}?{urlencode(params, doseq=True)}"

    fetched_at = now_utc()

    try:
        response = requests.get(
            url,
            params=params,
            headers=final_headers,
            timeout=timeout
        )

        response_text = response.text
        response_status_code = int(response.status_code)
        response_ok = bool(response.ok)

        try:
            payload = response.json()
        except Exception:
            payload = {"raw_text": response_text}

        error_message = None

    except Exception as e:
        response_status_code = None
        response_ok = False
        payload = None
        error_message = str(e)

    raw_payload_json = json_dumps_safe(payload)

    return normalize_external_row({
        "api_run_id": str(uuid.uuid4()),
        "ingestion_batch_id": INGESTION_BATCH_ID,
        "source_name": source_name,
        "endpoint_name": endpoint_name,
        "request_url": request_url,
        "query_params_json": json_dumps_safe(params),
        "request_headers_json": json_dumps_safe({
            k: ("***" if "key" in k.lower() or "authorization" in k.lower() else v)
            for k, v in final_headers.items()
        }),
        "response_status_code": response_status_code,
        "response_ok": response_ok,
        "fetched_at": fetched_at,
        "raw_payload_json": raw_payload_json,
        "payload_hash": sha256_text(raw_payload_json),
        "error_message": error_message,
        "ingestion_version": INGESTION_VERSION,
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })


def write_rows_to_bronze(rows, table_name):
    if not rows:
        print(f"No rows to write for {table_name}")
        return

    normalized_rows = [normalize_external_row(row) for row in rows]

    df = spark.createDataFrame(normalized_rows, schema=EXTERNAL_RAW_SCHEMA)

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"{BRONZE_SCHEMA}.{table_name}")
    )

    print(f"Wrote {len(rows)} rows to {BRONZE_SCHEMA}.{table_name}")


def append_api_runs(rows):
    api_run_rows = []

    for r in rows:
        api_run_rows.append({
            "api_run_id": r.get("api_run_id"),
            "ingestion_batch_id": r.get("ingestion_batch_id"),
            "source_name": r.get("source_name"),
            "endpoint_name": r.get("endpoint_name"),
            "request_url": r.get("request_url"),
            "response_status_code": r.get("response_status_code"),
            "response_ok": r.get("response_ok"),
            "fetched_at": r.get("fetched_at"),
            "payload_hash": r.get("payload_hash"),
            "error_message": r.get("error_message"),
            "ingestion_version": r.get("ingestion_version"),
            "created_by": r.get("created_by"),
            "notebook_name": r.get("notebook_name")
        })

    normalized_rows = []

    for row in api_run_rows:
        normalized_rows.append({
            "api_run_id": str(row.get("api_run_id")) if row.get("api_run_id") is not None else None,
            "ingestion_batch_id": str(row.get("ingestion_batch_id")) if row.get("ingestion_batch_id") is not None else None,
            "source_name": str(row.get("source_name")) if row.get("source_name") is not None else None,
            "endpoint_name": str(row.get("endpoint_name")) if row.get("endpoint_name") is not None else None,
            "request_url": str(row.get("request_url")) if row.get("request_url") is not None else None,
            "response_status_code": int(row.get("response_status_code")) if row.get("response_status_code") is not None else None,
            "response_ok": bool(row.get("response_ok")) if row.get("response_ok") is not None else None,
            "fetched_at": str(row.get("fetched_at")) if row.get("fetched_at") is not None else None,
            "payload_hash": str(row.get("payload_hash")) if row.get("payload_hash") is not None else None,
            "error_message": str(row.get("error_message")) if row.get("error_message") is not None else None,
            "ingestion_version": str(row.get("ingestion_version")) if row.get("ingestion_version") is not None else None,
            "created_by": str(row.get("created_by")) if row.get("created_by") is not None else None,
            "notebook_name": str(row.get("notebook_name")) if row.get("notebook_name") is not None else None
        })

    df = spark.createDataFrame(normalized_rows, schema=API_RUN_SCHEMA)

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"{BRONZE_SCHEMA}.bronze_ext_api_runs")
    )

    print(f"Wrote {len(rows)} rows to {BRONZE_SCHEMA}.bronze_ext_api_runs")

In [0]:
GDELT_BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

gdelt_queries = [
    "supply chain disruption",
    "port strike shipping delay",
    "factory shutdown supplier",
    "tariff retail supply chain",
    "logistics disruption retailer",
    "weather disruption supply chain"
]

gdelt_rows = []

for query in gdelt_queries:
    params = {
        "query": query,
        "mode": "artlist",
        "format": "json",
        "maxrecords": 50,
        "sort": "hybridrel"
    }

    row = safe_get_json(
        source_name="gdelt",
        endpoint_name="doc_artlist",
        url=GDELT_BASE_URL,
        params=params
    )

    gdelt_rows.append(row)
    time.sleep(1)

write_rows_to_bronze(gdelt_rows, "bronze_ext_gdelt_doc_raw")
append_api_runs(gdelt_rows)

display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_gdelt_doc_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .select(
        "source_name",
        "endpoint_name",
        "response_status_code",
        "response_ok",
        "fetched_at",
        "error_message"
    )
    .orderBy(F.col("fetched_at").desc())
)

In [0]:
import requests

test_url = "https://api.weather.gov/alerts/active"
test_params = {"area": "TX"}

test_response = requests.get(
    test_url,
    params=test_params,
    headers={
        "User-Agent": USER_AGENT,
        "Accept": "application/geo+json"
    },
    timeout=60
)

print("Status code:", test_response.status_code)
print("OK:", test_response.ok)
print("Content type:", test_response.headers.get("Content-Type"))
print("First 500 chars:")
print(test_response.text[:500])

In [0]:
NWS_BASE_URL = "https://api.weather.gov/alerts/active"

nws_areas = ["CA", "TX", "WI"]

nws_rows = []

for area in nws_areas:
    params = {
        "area": area
    }

    row = safe_get_json(
        source_name="nws",
        endpoint_name="active_alerts_by_area",
        url=NWS_BASE_URL,
        params=params,
        headers={
            "Accept": "application/geo+json"
        }
    )

    nws_rows.append(row)
    time.sleep(1)

write_rows_to_bronze(nws_rows, "bronze_ext_nws_alerts_raw")
append_api_runs(nws_rows)

display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_nws_alerts_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .select(
        "source_name",
        "endpoint_name",
        "response_status_code",
        "response_ok",
        "fetched_at",
        "error_message"
    )
    .orderBy(F.col("fetched_at").desc())
)

In [0]:
CISA_KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

cisa_rows = [
    safe_get_json(
        source_name="cisa",
        endpoint_name="known_exploited_vulnerabilities_catalog",
        url=CISA_KEV_URL,
        params={}
    )
]

write_rows_to_bronze(cisa_rows, "bronze_ext_cisa_kev_raw")
append_api_runs(cisa_rows)

display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_cisa_kev_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .select(
        "source_name",
        "endpoint_name",
        "response_status_code",
        "response_ok",
        "fetched_at",
        "error_message"
    )
    .orderBy(F.col("fetched_at").desc())
)

In [0]:
CPSC_RECALLS_URL = "https://www.saferproducts.gov/RestWebServices/Recall"

cpsc_rows = [
    safe_get_json(
        source_name="cpsc",
        endpoint_name="recalls_json",
        url=CPSC_RECALLS_URL,
        params={
            "format": "json"
        }
    )
]

write_rows_to_bronze(cpsc_rows, "bronze_ext_cpsc_recalls_raw")
append_api_runs(cpsc_rows)

display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_cpsc_recalls_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .select(
        "source_name",
        "endpoint_name",
        "response_status_code",
        "response_ok",
        "fetched_at",
        "error_message"
    )
    .orderBy(F.col("fetched_at").desc())
)

In [0]:
def safe_get_text(source_name, endpoint_name, url, params=None, headers=None, timeout=90):
    if params is None:
        params = {}

    if headers is None:
        headers = {}

    final_headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/xml,text/xml,text/plain,*/*"
    }
    final_headers.update(headers)

    request_url = url
    if params:
        request_url = f"{url}?{urlencode(params, doseq=True)}"

    fetched_at = now_utc()

    try:
        response = requests.get(
            url,
            params=params,
            headers=final_headers,
            timeout=timeout
        )

        response_text = response.text
        response_status_code = int(response.status_code)
        response_ok = bool(response.ok)
        error_message = None

    except Exception as e:
        response_text = ""
        response_status_code = None
        response_ok = False
        error_message = str(e)

    payload = {
        "raw_text": response_text
    }

    raw_payload_json = json_dumps_safe(payload)

    return normalize_external_row({
        "api_run_id": str(uuid.uuid4()),
        "ingestion_batch_id": INGESTION_BATCH_ID,
        "source_name": source_name,
        "endpoint_name": endpoint_name,
        "request_url": request_url,
        "query_params_json": json_dumps_safe(params),
        "request_headers_json": json_dumps_safe(final_headers),
        "response_status_code": response_status_code,
        "response_ok": response_ok,
        "fetched_at": fetched_at,
        "raw_payload_json": raw_payload_json,
        "payload_hash": sha256_text(raw_payload_json),
        "error_message": error_message,
        "ingestion_version": INGESTION_VERSION,
        "created_by": CREATED_BY,
        "notebook_name": NOTEBOOK_NAME
    })


ofac_sources = [
    {
        "endpoint_name": "sdn_xml",
        "url": "https://www.treasury.gov/ofac/downloads/sdn.xml"
    },
    {
        "endpoint_name": "consolidated_xml",
        "url": "https://www.treasury.gov/ofac/downloads/consolidated/consolidated.xml"
    }
]

ofac_rows = []

for src in ofac_sources:
    row = safe_get_text(
        source_name="ofac",
        endpoint_name=src["endpoint_name"],
        url=src["url"]
    )

    ofac_rows.append(row)
    time.sleep(1)

write_rows_to_bronze(ofac_rows, "bronze_ext_ofac_sanctions_raw")
append_api_runs(ofac_rows)

display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_ofac_sanctions_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .select(
        "source_name",
        "endpoint_name",
        "response_status_code",
        "response_ok",
        "fetched_at",
        "error_message"
    )
    .orderBy(F.col("fetched_at").desc())
)

In [0]:
# Clean the SEC rows from this batch so we can rewrite them with company metadata.
spark.sql(f"""
DELETE FROM {BRONZE_SCHEMA}.bronze_ext_sec_company_submissions_raw
WHERE ingestion_batch_id = '{INGESTION_BATCH_ID}'
""")

spark.sql(f"""
DELETE FROM {BRONZE_SCHEMA}.bronze_ext_api_runs
WHERE ingestion_batch_id = '{INGESTION_BATCH_ID}'
  AND source_name = 'sec_edgar'
  AND endpoint_name = 'company_submissions'
""")

print("Deleted previous SEC rows for this batch.")


SEC_RAW_SCHEMA = T.StructType(EXTERNAL_RAW_SCHEMA.fields + [
    T.StructField("company_name", T.StringType(), True),
    T.StructField("ticker", T.StringType(), True),
    T.StructField("cik", T.StringType(), True)
])


def normalize_sec_row(row):
    base_row = normalize_external_row(row)

    base_row["company_name"] = str(row.get("company_name")) if row.get("company_name") is not None else None
    base_row["ticker"] = str(row.get("ticker")) if row.get("ticker") is not None else None
    base_row["cik"] = str(row.get("cik")) if row.get("cik") is not None else None

    return base_row


def write_sec_rows_to_bronze(rows, table_name):
    if not rows:
        print(f"No rows to write for {table_name}")
        return

    normalized_rows = [normalize_sec_row(row) for row in rows]

    df = spark.createDataFrame(normalized_rows, schema=SEC_RAW_SCHEMA)

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"{BRONZE_SCHEMA}.{table_name}")
    )

    print(f"Wrote {len(rows)} rows to {BRONZE_SCHEMA}.{table_name}")


SEC_SUBMISSIONS_BASE = "https://data.sec.gov/submissions"

sec_companies = [
    {"company_name": "Walmart", "ticker": "WMT", "cik": "0000104169"},
    {"company_name": "Target", "ticker": "TGT", "cik": "0000027419"},
    {"company_name": "Costco", "ticker": "COST", "cik": "0000909832"},
    {"company_name": "Amazon", "ticker": "AMZN", "cik": "0001018724"},
    {"company_name": "FedEx", "ticker": "FDX", "cik": "0001048911"},
    {"company_name": "UPS", "ticker": "UPS", "cik": "0001090727"}
]

sec_rows = []

for company in sec_companies:
    cik = company["cik"]
    url = f"{SEC_SUBMISSIONS_BASE}/CIK{cik}.json"

    row = safe_get_json(
        source_name="sec_edgar",
        endpoint_name="company_submissions",
        url=url,
        params={},
        headers={
            "User-Agent": USER_AGENT,
            "Accept-Encoding": "gzip, deflate",
            "Host": "data.sec.gov"
        }
    )

    row["company_name"] = company["company_name"]
    row["ticker"] = company["ticker"]
    row["cik"] = company["cik"]

    sec_rows.append(row)
    time.sleep(0.2)

write_sec_rows_to_bronze(sec_rows, "bronze_ext_sec_company_submissions_raw")
append_api_runs(sec_rows)

display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_sec_company_submissions_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .select(
        "source_name",
        "endpoint_name",
        "company_name",
        "ticker",
        "cik",
        "response_status_code",
        "response_ok",
        "fetched_at",
        "error_message"
    )
    .orderBy("company_name")
)

In [0]:
eia_rows = []

if EIA_API_KEY is None:
    print("Skipping EIA because EIA_API_KEY is missing.")
else:
    EIA_BASE_URL = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"

    eia_queries = [
        {
            "endpoint_name": "petroleum_prices_weekly_recent",
            "params": {
                "api_key": EIA_API_KEY,
                "frequency": "weekly",
                "data[0]": "value",
                "sort[0][column]": "period",
                "sort[0][direction]": "desc",
                "offset": 0,
                "length": 500
            }
        }
    ]

    for query in eia_queries:
        row = safe_get_json(
            source_name="eia",
            endpoint_name=query["endpoint_name"],
            url=EIA_BASE_URL,
            params=query["params"]
        )

        # Mask API key before storing request metadata.
        row["request_url"] = row["request_url"].replace(EIA_API_KEY, "***")

        query_params = json.loads(row["query_params_json"])
        if "api_key" in query_params:
            query_params["api_key"] = "***"
        row["query_params_json"] = json_dumps_safe(query_params)

        eia_rows.append(row)
        time.sleep(1)

    write_rows_to_bronze(eia_rows, "bronze_ext_eia_fuel_prices_raw")
    append_api_runs(eia_rows)

    display(
        spark.table(f"{BRONZE_SCHEMA}.bronze_ext_eia_fuel_prices_raw")
        .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
        .select(
            "source_name",
            "endpoint_name",
            "response_status_code",
            "response_ok",
            "fetched_at",
            "error_message"
        )
        .orderBy(F.col("fetched_at").desc())
    )

In [0]:
source_registry_rows = [
    {
        "source_name": "gdelt",
        "bronze_table": "bronze_ext_gdelt_doc_raw",
        "risk_category": "news_disruption",
        "requires_api_key": False,
        "business_purpose": "Global news signals for supply chain disruption, strikes, tariffs, weather disruption, and logistics risk.",
        "included_in_v0": True
    },
    {
        "source_name": "nws",
        "bronze_table": "bronze_ext_nws_alerts_raw",
        "risk_category": "weather_logistics",
        "requires_api_key": False,
        "business_purpose": "Official weather alerts for M5 states CA, TX, and WI.",
        "included_in_v0": True
    },
    {
        "source_name": "cisa",
        "bronze_table": "bronze_ext_cisa_kev_raw",
        "risk_category": "cyber",
        "requires_api_key": False,
        "business_purpose": "Known exploited vulnerabilities catalog for supplier cyber-risk evidence.",
        "included_in_v0": True
    },
    {
        "source_name": "cpsc",
        "bronze_table": "bronze_ext_cpsc_recalls_raw",
        "risk_category": "quality_recall",
        "requires_api_key": False,
        "business_purpose": "Consumer product recalls for quality and product-safety risk.",
        "included_in_v0": True
    },
    {
        "source_name": "ofac",
        "bronze_table": "bronze_ext_ofac_sanctions_raw",
        "risk_category": "sanctions_compliance",
        "requires_api_key": False,
        "business_purpose": "Sanctions list snapshots for later supplier compliance screening.",
        "included_in_v0": True
    },
    {
        "source_name": "sec_edgar",
        "bronze_table": "bronze_ext_sec_company_submissions_raw",
        "risk_category": "financial_filing",
        "requires_api_key": False,
        "business_purpose": "SEC company submissions metadata for later financial and filing-risk analysis.",
        "included_in_v0": True
    },
    {
        "source_name": "eia",
        "bronze_table": "bronze_ext_eia_fuel_prices_raw",
        "risk_category": "fuel_logistics_cost",
        "requires_api_key": True,
        "business_purpose": "Fuel price data for transportation and logistics-cost risk signals.",
        "included_in_v0": EIA_API_KEY is not None
    }
]

registry_schema = T.StructType([
    T.StructField("source_name", T.StringType(), True),
    T.StructField("bronze_table", T.StringType(), True),
    T.StructField("risk_category", T.StringType(), True),
    T.StructField("requires_api_key", T.BooleanType(), True),
    T.StructField("business_purpose", T.StringType(), True),
    T.StructField("included_in_v0", T.BooleanType(), True)
])

registry_df = spark.createDataFrame(source_registry_rows, schema=registry_schema)

(
    registry_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{BRONZE_SCHEMA}.bronze_ext_source_registry")
)

display(registry_df.orderBy("source_name"))

In [0]:
api_runs_df = spark.table(f"{BRONZE_SCHEMA}.bronze_ext_api_runs")

display(
    api_runs_df
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .groupBy("source_name")
    .agg(
        F.count("*").alias("api_call_count"),
        F.sum(F.when(F.col("response_ok") == True, 1).otherwise(0)).alias("successful_calls"),
        F.sum(F.when(F.col("response_ok") == False, 1).otherwise(0)).alias("failed_calls"),
        F.collect_set("response_status_code").alias("status_codes")
    )
    .orderBy("source_name")
)

In [0]:
GDELT_BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

# Smaller, gentler v0 query set to avoid 429 rate limits
gdelt_queries = [
    "supply chain disruption",
    "port strike shipping delay"
]

gdelt_rows = []

for query in gdelt_queries:
    params = {
        "query": query,
        "mode": "artlist",
        "format": "json",
        "maxrecords": 20,
        "sort": "hybridrel"
    }

    row = safe_get_json(
        source_name="gdelt",
        endpoint_name="doc_artlist",
        url=GDELT_BASE_URL,
        params=params
    )

    # If rate-limited, wait and retry once
    if row["response_status_code"] == 429:
        print(f"GDELT rate-limited for query: {query}. Waiting 20 seconds and retrying once.")
        time.sleep(20)

        row = safe_get_json(
            source_name="gdelt",
            endpoint_name="doc_artlist",
            url=GDELT_BASE_URL,
            params=params
        )

    gdelt_rows.append(row)
    time.sleep(10)

write_rows_to_bronze(gdelt_rows, "bronze_ext_gdelt_doc_raw")
append_api_runs(gdelt_rows)

display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_gdelt_doc_raw")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .select(
        "source_name",
        "endpoint_name",
        "response_status_code",
        "response_ok",
        "fetched_at",
        "error_message"
    )
    .orderBy(F.col("fetched_at").desc())
)

In [0]:
spark.sql(f"""
DELETE FROM {BRONZE_SCHEMA}.bronze_ext_gdelt_doc_raw
WHERE ingestion_batch_id = '{INGESTION_BATCH_ID}'
  AND source_name = 'gdelt'
  AND response_ok = false
""")

spark.sql(f"""
DELETE FROM {BRONZE_SCHEMA}.bronze_ext_api_runs
WHERE ingestion_batch_id = '{INGESTION_BATCH_ID}'
  AND source_name = 'gdelt'
  AND response_ok = false
""")

print("Deleted failed GDELT rows for this batch. Kept successful GDELT rows.")

In [0]:
display(
    spark.table(f"{BRONZE_SCHEMA}.bronze_ext_api_runs")
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .groupBy("source_name")
    .agg(
        F.count("*").alias("api_call_count"),
        F.sum(F.when(F.col("response_ok") == True, 1).otherwise(0)).alias("successful_calls"),
        F.sum(F.when(F.col("response_ok") == False, 1).otherwise(0)).alias("failed_calls"),
        F.collect_set("response_status_code").alias("status_codes")
    )
    .orderBy("source_name")
)

In [0]:
required_sources = ["gdelt", "nws", "cisa", "cpsc", "ofac", "sec_edgar", "eia"]

api_runs_df = spark.table(f"{BRONZE_SCHEMA}.bronze_ext_api_runs")

final_validation_df = (
    api_runs_df
    .filter(F.col("ingestion_batch_id") == INGESTION_BATCH_ID)
    .filter(F.col("source_name").isin(required_sources))
    .groupBy("source_name")
    .agg(
        F.count("*").alias("api_call_count"),
        F.sum(F.when(F.col("response_ok") == True, 1).otherwise(0)).alias("successful_calls"),
        F.sum(F.when(F.col("response_ok") == False, 1).otherwise(0)).alias("failed_calls"),
        F.collect_set("response_status_code").alias("status_codes")
    )
    .orderBy("source_name")
)

display(final_validation_df)

successful_source_count = (
    final_validation_df
    .filter(F.col("successful_calls") >= 1)
    .count()
)

failed_source_count = (
    final_validation_df
    .filter(F.col("failed_calls") > 0)
    .count()
)

total_source_count = len(required_sources)

print("Required sources:", total_source_count)
print("Sources with at least one successful call:", successful_source_count)
print("Sources with failed calls:", failed_source_count)

if successful_source_count == total_source_count and failed_source_count == 0:
    print("✅ Notebook 05 PASSED: External raw Bronze ingestion completed cleanly.")
    print("Next notebook: 06_bronze_validate_external_risk_sources")
else:
    print("⚠️ Notebook 05 needs review before moving on.")